# K-Means Konvensional vs Metode Usulan pada CIFAR-10

Notebook ini menjalankan **K-Means konvensional** dan **metode usulan** (objek simbolik + FDKE per kanal + divergensi Jensen-Shannon + inisialisasi Cartesian Join + FDK multi-intensitas) pada **data dan seed yang sama**, lalu menghitung studi ablasi dan uji t berpasangan.

| Varian | Representasi | Jarak | Inisialisasi |
|---|---|---|---|
| **A — Konvensional** | rata-rata R, G, B (3 fitur) | Euclidean kuadrat | acak (Forgy) |
| **B** | simbolik ringkas 17 komponen: rata-rata, selang [min, maks] R, G, B, dan fitur tepi Canny | Euclidean kuadrat | acak |
| **C** | B + FDKE 10 bin per kanal | Jensen-Shannon (FDKE) + Euclidean (fitur numerik) | acak |
| **D** | C | idem | Cartesian Join |
| **E — Metode usulan** | D + FDK multi-intensitas (v = 100, 125, 150) | idem | Cartesian Join |

**Sumber data.** Gambar CIFAR-10 dibaca **langsung dari folder lokal** (tanpa unduh):

    C:\Users\Ahsan\Downloads\data\archive\cifar10_train\<kelas>\*.png   (5.000 per kelas)
    C:\Users\Ahsan\Downloads\data\archive\cifar10_test\<kelas>\*.png    (1.000 per kelas)

Berkas PNG berukuran 32 × 32 piksel RGB sehingga isinya setara dengan larik `datasets.CIFAR10`. **Urutan berkas** di notebook ini adalah per kelas lalu indeks numerik pada nama berkas (`airplane_00003.png` → indeks 3), berbeda dari urutan batch resmi CIFAR-10. Akibatnya himpunan citra yang dipakai sama, tetapi (a) urutan indeks berbeda dan (b) subsampel acak untuk Silhouette berbeda, sehingga angka **per seed** tidak identik dengan versi Colab yang memakai `torchvision`; statistik rata-rata 30 seed masih sebanding dan harus dilaporkan berdasarkan run ini.

**Skema evaluasi** (Subbab III.4.3 dan IV.1.1.1):
- 10.000 citra uji CIFAR-10 (dari folder lokal `cifar10_test`);
- 30 pengulangan dengan seed acak;
- akurasi setelah pemetaan Hungarian, purity, ARI, NMI, V-Measure, Silhouette, Davies-Bouldin, Calinski-Harabasz;
- waktu, jumlah iterasi, dan memori puncak.

**Catatan.** Naskah disertasi tidak merinci bobot antara bagian distribusi dan bagian numerik, ambang Canny, maupun cara memilih objek acuan Cartesian Join tanpa label. Nilai-nilai tersebut dijadikan parameter di sel Konfigurasi (`LAMBDA_NUM`, `CANNY_LOW/HIGH`, `CJ_MODE`, `CJ_M`), sehingga hasil notebook ini bisa berbeda dari angka di buku.

## 1. Impor pustaka

In [ ]:
import os, time, random, tracemalloc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score, v_measure_score,
                             silhouette_score, davies_bouldin_score, calinski_harabasz_score)

## 2. Konfigurasi

In [ ]:
K            = 10            # banyak klaster
N_RUNS       = 30            # banyak pengulangan (seed acak berbeda)
MASTER_SEED  = 2026          # agar daftar 30 seed dapat direproduksi
MAX_ITER     = 100           # batas iterasi K-Means
N_BINS       = 10            # bin FDKE per kanal
FDK_INTENS   = [100, 125, 150]   # intensitas target FDK multi-intensitas (Subbab IV.5.4)
CANNY_LOW, CANNY_HIGH = 100, 200 # ambang deteksi tepi Canny
LAMBDA_NUM   = 1.0           # bobot bagian numerik pada jarak gabungan
CJ_MODE      = "unsupervised"    # "unsupervised": m tetangga terdekat dari benih K-Means++ (tanpa label)
                                 # "seeded": m citra latih per kelas (memakai label, seperti Subbab IV.3.2)
CJ_M         = 5             # banyak objek acuan per klaster untuk Cartesian Join
SIL_SAMPLE   = 3000          # ukuran sampel Silhouette (hemat waktu)
VARIANTS     = ["A", "B", "C", "D", "E"]

# --- Data: gambar CIFAR-10 lokal, tanpa unduh --------------------------------
ARCHIVE_DIR = Path(r"C:\Users\Ahsan\Downloads\data\archive")
TRAIN_DIR   = ARCHIVE_DIR / "cifar10_train"   # cifar10_train/<kelas>/*.png, 5.000 per kelas
TEST_DIR    = ARCHIVE_DIR / "cifar10_test"    # cifar10_test/<kelas>/*.png,  1.000 per kelas
IMG_SIZE    = 32             # PNG sudah 32x32; resize hanya dilakukan bila ukurannya lain
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

# --- Keluaran: langsung ke folder Downloads ---------------------------------
OUTPUT_DIR   = Path(r"C:\Users\Ahsan\Downloads")
OUTPUT_EXCEL = OUTPUT_DIR / "cifar10_konvensional_vs_usulan.xlsx"
OUTPUT_PNG   = OUTPUT_DIR / "cifar10_ablasi.png"

for _d in (TRAIN_DIR, TEST_DIR):
    if not _d.is_dir():
        raise FileNotFoundError(f"Folder data tidak ditemukan: {_d}. Periksa ARCHIVE_DIR di sel ini.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data latih :", TRAIN_DIR)
print("Data uji   :", TEST_DIR)
print("Keluaran   :", OUTPUT_DIR)

random.seed(MASTER_SEED)
SEEDS = [random.randint(1, 10000) for _ in range(N_RUNS)]
print("Seed:", SEEDS)

## 3. Memuat data CIFAR-10 dari folder lokal

Gambar dibaca dari `cifar10_train/<kelas>/*.png` dan `cifar10_test/<kelas>/*.png`, tanpa mengunduh apa pun. Label kelas diambil dari nama folder mengikuti urutan `CLASS_NAMES`, jadi label integer selalu 0 = airplane … 9 = truck.

Data latih dipakai **tanpa label** untuk membangun model FDK (peringkat persentil FDKE). Labelnya hanya dipakai bila `CJ_MODE = "seeded"`.

In [ ]:
def _file_key(path):
    """Kunci urut alami: 'airplane_00003.png' -> ('airplane', 3)."""
    stem = Path(path).stem
    head, _, tail = stem.rpartition("_")
    return (head, int(tail)) if tail.isdigit() else (stem, -1)

def load_split(split_dir, class_names=CLASS_NAMES):
    """Muat satu split dari <split_dir>/<kelas>/*.png -> (X uint8 RGB, y, nama berkas)."""
    X, y, berkas = [], [], []
    for ci, cname in enumerate(class_names):
        folder = Path(split_dir) / cname
        files = sorted(folder.glob("*.png"), key=_file_key)
        if not files:
            raise FileNotFoundError(f"Tidak ada berkas PNG untuk kelas '{cname}' di {folder}")
        for f in files:
            im = cv2.imread(str(f), cv2.IMREAD_COLOR)          # OpenCV memuat BGR
            if im is None:
                raise IOError(f"Gagal membaca berkas: {f}")
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)           # samakan dengan CIFAR-10 (RGB)
            if im.shape[:2] != (IMG_SIZE, IMG_SIZE):
                im = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            X.append(im); y.append(ci); berkas.append(f.name)
    return np.stack(X).astype(np.uint8), np.asarray(y, dtype=np.int64), np.asarray(berkas)

def check_class_folders(split_dir):
    """Peringatkan bila ada subfolder yang tidak ada di CLASS_NAMES (diabaikan)."""
    extra = sorted({p.name for p in Path(split_dir).iterdir() if p.is_dir()} - set(CLASS_NAMES))
    if extra:
        print(f"Peringatan: subfolder tak dikenal di {Path(split_dir).name} (diabaikan): {extra}")

t0 = time.perf_counter()
for _d in (TRAIN_DIR, TEST_DIR):
    check_class_folders(_d)
X_train_img, y_train, TRAIN_FILES = load_split(TRAIN_DIR)
X_test_img,  y_test,  TEST_FILES  = load_split(TEST_DIR)
print(f"Memuat {len(TRAIN_FILES) + len(TEST_FILES)} PNG dalam {time.perf_counter() - t0:.1f} detik")
print("Latih:", X_train_img.shape, X_train_img.dtype, " Uji:", X_test_img.shape, X_test_img.dtype)
print("Citra per kelas (latih):", np.bincount(y_train, minlength=K).tolist())
print("Citra per kelas (uji)  :", np.bincount(y_test, minlength=K).tolist())
print("Contoh berkas uji      :", TEST_FILES[:3].tolist())

## 4. Ekstraksi fitur

* **Rata-rata RGB**: rata-rata tiap kanal atas 1.024 piksel, dibagi 255.
* **Simbolik ringkas (17 komponen)**: rata-rata R, G, B; selang [min, maks] R, G, B (dibagi 255); dari piksel tepi Canny: posisi rata-rata (x̄, ȳ), selang [min, maks] x dan y, serta panjang sebaran Length x, Length y (dibagi 32).
* **FDKE per kanal**: histogram 10 bin dinormalisasi menjadi peluang per bin. FDKE adalah jumlah kumulatifnya; divergensi Jensen-Shannon dihitung pada peluang per bin.
* **FDK multi-intensitas**: untuk setiap kanal c dan intensitas v, nilai FDKE citra F_c(v) dipetakan ke peringkat persentilnya Ψ_{c,v} di antara citra latih, sehingga diperoleh 3 × 3 = 9 komponen pada [0, 1].

In [ ]:
def rgb_mean(imgs):
    return imgs.reshape(len(imgs), -1, 3).mean(axis=1) / 255.0

def symbolic17(imgs, low=CANNY_LOW, high=CANNY_HIGH):
    n, p = len(imgs), imgs.shape[1]
    flat = imgs.reshape(n, -1, 3).astype(np.float64)
    mean, mn, mx = flat.mean(1) / 255, flat.min(1) / 255, flat.max(1) / 255
    edge = np.zeros((n, 8))
    for i, im in enumerate(imgs):
        gray = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
        ys, xs = np.nonzero(cv2.Canny(gray, low, high))
        if len(xs) == 0:                       # tidak ada tepi: pakai pusat citra
            edge[i] = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0, 0]
            continue
        x, y = xs / p, ys / p
        edge[i] = [x.mean(), y.mean(), x.min(), x.max(), y.min(), y.max(),
                   x.max() - x.min(), y.max() - y.min()]
    # urutan: mean R,G,B | min R,max R,min G,max G,min B,max B | x̄, ȳ | min x,max x,min y,max y | Lx, Ly
    inter = np.column_stack([mn[:, 0], mx[:, 0], mn[:, 1], mx[:, 1], mn[:, 2], mx[:, 2]])
    return np.hstack([mean, inter, edge[:, :2], edge[:, 2:6], edge[:, 6:]])

# indeks komponen bernilai selang (pasangan min, maks) di dalam vektor 17 komponen
INTERVAL_PAIRS = [(3, 4), (5, 6), (7, 8), (11, 12), (13, 14)]

def fdke_pdf(imgs, bins=N_BINS):
    n = len(imgs)
    out = np.zeros((n, 3, bins))
    edges = np.linspace(0, 256, bins + 1)
    flat = imgs.reshape(n, -1, 3)
    for c in range(3):
        idx = np.clip(np.digitize(flat[:, :, c], edges) - 1, 0, bins - 1)
        for b in range(bins):
            out[:, c, b] = (idx == b).mean(axis=1)
    return out                                   # (n, 3, bins); FDKE = np.cumsum(..., axis=2)

def fdke_at(imgs, v):
    # F_c(v): proporsi piksel kanal c dengan intensitas <= v
    return (imgs.reshape(len(imgs), -1, 3) <= v).mean(axis=1)    # (n, 3)

def fdk_multi(imgs, ref_imgs, intens=FDK_INTENS):
    cols = []
    for v in intens:
        F, Fref = fdke_at(imgs, v), fdke_at(ref_imgs, v)
        for c in range(3):
            s = np.sort(Fref[:, c])
            cols.append(np.searchsorted(s, F[:, c], side="right") / len(s))
    return np.column_stack(cols)                 # (n, 9)

t0 = time.perf_counter()
FEAT = {}
for name, imgs in [("test", X_test_img), ("train", X_train_img)]:
    FEAT[name] = dict(rgb=rgb_mean(imgs), sym=symbolic17(imgs), pdf=fdke_pdf(imgs),
                      fdk=fdk_multi(imgs, X_train_img))
print(f"Ekstraksi fitur selesai dalam {time.perf_counter() - t0:.1f} detik")
print({k: v.shape for k, v in FEAT["test"].items()})

## 5. Ukuran jarak

* Euclidean kuadrat untuk bagian numerik.
* Divergensi Jensen-Shannon (logaritma basis 2, sehingga 0 ≤ D_JS ≤ 1) untuk peluang per bin setiap kanal, dijumlahkan atas tiga kanal.
* Jarak gabungan: $D(i,k) = \sum_c D_{JS}(P_{ic}, Q_{kc}) + \lambda \,\lVert f_i - z_k \rVert^2$.

In [ ]:
def sqeuclid(X, Z):
    return ((X[:, None, :] - Z[None, :, :]) ** 2).sum(axis=2)

def _xlogx_ratio(P, M):
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(P > 0, P * np.log2(P / M), 0.0)
    return r

def js_matrix(P, Q):
    # P: (n, 3, b), Q: (k, 3, b) -> (n, k), jumlah D_JS atas tiga kanal
    Pn, Qk = P[:, None, :, :], Q[None, :, :, :]
    M = 0.5 * (Pn + Qk)
    js = 0.5 * _xlogx_ratio(Pn, M) + 0.5 * _xlogx_ratio(Qk, M)
    return js.sum(axis=(2, 3))

def dist_matrix(obj, cen, use_pdf, lam=LAMBDA_NUM):
    D = lam * sqeuclid(obj["num"], cen["num"]) if use_pdf else sqeuclid(obj["num"], cen["num"])
    if use_pdf:
        D = D + js_matrix(obj["pdf"], cen["pdf"])
    return D

## 6. Representasi setiap varian

In [ ]:
def make_objects(split, variant):
    f = FEAT[split]
    if variant == "A":
        return dict(num=f["rgb"], pdf=None), False
    if variant in ("B",):
        return dict(num=f["sym"], pdf=None), False
    if variant in ("C", "D"):
        return dict(num=f["sym"], pdf=f["pdf"]), True
    if variant == "E":
        return dict(num=np.hstack([f["sym"], f["fdk"]]), pdf=f["pdf"]), True
    raise ValueError(variant)

def take(obj, idx):
    return {k: (None if v is None else v[idx]) for k, v in obj.items()}

def flat_features(obj):
    # ruang fitur untuk metrik internal (Silhouette, DBI, CH)
    parts = [obj["num"]] + ([obj["pdf"].reshape(len(obj["num"]), -1)] if obj["pdf"] is not None else [])
    return np.hstack(parts)

## 7. Inisialisasi sentroid

* **Acak (Forgy)**: K objek dipilih acak dari data uji.
* **Cartesian Join (⊕)** atas m objek acuan per klaster (Subbab III.3.4):
  * atribut bernilai selang → selang gabungan [min dari semua min, maks dari semua maks];
  * atribut bernilai tunggal, FDKE, dan FDK → rata-rata objek acuan.
* Objek acuan:
  * `unsupervised`: benih dipilih dengan K-Means++ pada data uji, lalu diambil m tetangga terdekatnya (termasuk benih itu sendiri); tidak memakai label.
  * `seeded`: m citra latih acak dari setiap kelas; memakai label, seperti Subbab IV.3.2.

In [ ]:
def cartesian_join(ref, interval_pairs):
    num = ref["num"].mean(axis=0)
    for lo, hi in interval_pairs:
        num[lo] = ref["num"][:, lo].min()
        num[hi] = ref["num"][:, hi].max()
    pdf = None if ref["pdf"] is None else ref["pdf"].mean(axis=0)
    return num, pdf

def kmeanspp_idx(obj, k, rng, use_pdf):
    n = len(obj["num"])
    idx = [int(rng.integers(n))]
    d = dist_matrix(obj, take(obj, idx), use_pdf)[:, 0]
    for _ in range(1, k):
        p = np.maximum(d, 0); p = p / p.sum()          # d sudah berbentuk kuadrat/divergensi
        idx.append(int(rng.choice(n, p=p)))
        d = np.minimum(d, dist_matrix(obj, take(obj, [idx[-1]]), use_pdf)[:, 0])
    return idx

def init_centroids(obj, variant, use_pdf, rng):
    n = len(obj["num"])
    if variant in ("A", "B", "C"):
        idx = rng.choice(n, size=K, replace=False)
        return take(obj, idx)
    nums, pdfs = [], []
    if CJ_MODE == "seeded":
        tr, _ = make_objects("train", variant)
        for k in range(K):
            ref_idx = rng.choice(np.where(y_train == k)[0], size=CJ_M, replace=False)
            a, b = cartesian_join(take(tr, ref_idx), INTERVAL_PAIRS)
            nums.append(a); pdfs.append(b)
    else:
        seeds = kmeanspp_idx(obj, K, rng, use_pdf)
        Dseed = dist_matrix(obj, take(obj, seeds), use_pdf)        # (n, K)
        for k in range(K):
            ref_idx = np.argsort(Dseed[:, k])[:CJ_M]
            a, b = cartesian_join(take(obj, ref_idx), INTERVAL_PAIRS)
            nums.append(a); pdfs.append(b)
    return dict(num=np.array(nums), pdf=None if pdfs[0] is None else np.array(pdfs))

## 8. K-Means dengan pembaruan sentroid bersyarat

Mengikuti Algoritma 2 dan Teorema III.3.T1:
1. **Penugasan**: setiap citra ke sentroid terdekat menurut D.
2. **Pembaruan**: rata-rata anggota menjadi *calon* sentroid dan diterima hanya bila jumlah jarak anggota klaster turun. Untuk Euclidean kuadrat, rata-rata selalu diterima (sentroid eksak).
3. Klaster kosong mempertahankan sentroid lamanya.
4. Berhenti bila penugasan tidak berubah dan tidak ada sentroid yang berubah.

In [ ]:
def kmeans(obj, cen, use_pdf, max_iter=MAX_ITER):
    cen = {k: (None if v is None else v.copy()) for k, v in cen.items()}
    labels, n_iter = None, 0
    for t in range(max_iter):
        n_iter = t + 1
        D = dist_matrix(obj, cen, use_pdf)
        new_labels = D.argmin(axis=1)
        changed_assign = labels is None or not np.array_equal(new_labels, labels)
        labels = new_labels
        changed_cen = False
        for k in range(K):
            mem = np.where(labels == k)[0]
            if len(mem) == 0:
                continue
            sub = take(obj, mem)
            cand = dict(num=sub["num"].mean(axis=0, keepdims=True),
                        pdf=None if sub["pdf"] is None else sub["pdf"].mean(axis=0, keepdims=True))
            old = take(cen, [k])
            if use_pdf:
                if dist_matrix(sub, cand, True).sum() >= dist_matrix(sub, old, True).sum():
                    continue
            if not np.allclose(cand["num"][0], cen["num"][k]) or (
                    use_pdf and not np.allclose(cand["pdf"][0], cen["pdf"][k])):
                changed_cen = True
            cen["num"][k] = cand["num"][0]
            if use_pdf:
                cen["pdf"][k] = cand["pdf"][0]
        if not changed_assign and not changed_cen:
            break
    J = float(dist_matrix(obj, cen, use_pdf)[np.arange(len(labels)), labels].sum())
    return labels, cen, n_iter, J

## 9. Evaluasi

In [ ]:
def confusion(y_true, y_pred):
    cm = np.zeros((K, K), dtype=np.int64)
    np.add.at(cm, (y_pred, y_true), 1)
    return cm

def evaluate(Xflat, y_true, labels, seed):
    cm = confusion(y_true, labels)
    r, c = linear_sum_assignment(-cm)
    nclu = len(np.unique(labels))
    return dict(
        accuracy=cm[r, c].sum() / len(y_true),
        purity=cm.max(axis=1).sum() / len(y_true),
        ARI=adjusted_rand_score(y_true, labels),
        NMI=normalized_mutual_info_score(y_true, labels),
        V_measure=v_measure_score(y_true, labels),
        silhouette=silhouette_score(Xflat, labels, sample_size=min(SIL_SAMPLE, len(labels)),
                                    random_state=seed) if nclu > 1 else np.nan,
        DBI=davies_bouldin_score(Xflat, labels) if nclu > 1 else np.nan,
        CH=calinski_harabasz_score(Xflat, labels) if nclu > 1 else np.nan,
    ), dict(zip(r, c))

## 10. Menjalankan semua varian dengan 30 seed yang sama

In [ ]:
records, best = [], {}
for v in VARIANTS:
    obj, use_pdf = make_objects("test", v)
    Xflat = flat_features(obj)
    t_var = time.perf_counter()
    for run, seed in enumerate(SEEDS, start=1):
        rng = np.random.default_rng(seed)
        tracemalloc.start()
        t0 = time.perf_counter()
        cen0 = init_centroids(obj, v, use_pdf, rng)
        labels, cen, n_iter, J = kmeans(obj, cen0, use_pdf)
        elapsed = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        m, mapping = evaluate(Xflat, y_test, labels, seed)
        records.append(dict(variant=v, run=run, seed=seed, **m, objective=J, n_iter=n_iter,
                            time_s=elapsed, memory_peak_MB=peak / 2**20))
        if v not in best or J < best[v]["J"]:
            best[v] = dict(J=J, labels=labels, mapping=mapping)
    print(f"Varian {v} selesai ({time.perf_counter() - t_var:.1f} s); "
          f"akurasi rata-rata {np.mean([r['accuracy'] for r in records if r['variant'] == v]):.4f}")

df = pd.DataFrame(records)

## 11. Ringkasan dan studi ablasi

In [ ]:
METRICS = ["accuracy", "purity", "ARI", "NMI", "V_measure", "silhouette", "DBI", "CH",
           "n_iter", "time_s", "memory_peak_MB"]
summary = df.groupby("variant")[METRICS].agg(["mean", "std"])
summary.columns = [f"{a}_{b}" for a, b in summary.columns]

LABEL = {"A": "A  Konvensional (rata-rata RGB)", "B": "B  Simbolik ringkas 17 komponen",
         "C": "C  + FDKE (Jensen-Shannon)", "D": "D  + Cartesian Join",
         "E": "E  + FDK multi-intensitas (usulan)"}
tab = pd.DataFrame({
    "Varian": [LABEL[v] for v in VARIANTS],
    "Akurasi (%)": [f"{summary.loc[v,'accuracy_mean']*100:.2f} ± {summary.loc[v,'accuracy_std']*100:.2f}" for v in VARIANTS],
    "ARI": [f"{summary.loc[v,'ARI_mean']:.3f}" for v in VARIANTS],
    "NMI": [f"{summary.loc[v,'NMI_mean']:.3f}" for v in VARIANTS],
    "DBI": [f"{summary.loc[v,'DBI_mean']:.3f}" for v in VARIANTS],
    "Iterasi": [f"{summary.loc[v,'n_iter_mean']:.1f}" for v in VARIANTS],
    "Waktu (s)": [f"{summary.loc[v,'time_s_mean']:.2f}" for v in VARIANTS],
})
display(tab)

## 12. Uji t berpasangan satu sisi (H₁: varian lebih akurat daripada konvensional A)

In [ ]:
piv = df.pivot(index="seed", columns="variant", values="accuracy")
rows = []
for v in VARIANTS[1:]:
    d = piv[v] - piv["A"]
    t, p = stats.ttest_rel(piv[v], piv["A"], alternative="greater")
    rows.append({"Pembanding": f"{v} vs A", "Selisih rata-rata (pp)": d.mean() * 100,
                 "SD selisih (pp)": d.std(ddof=1) * 100, "t": t, "p": p,
                 "Cohen's d": d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan})
ttest = pd.DataFrame(rows)
display(ttest.round(4))

## 13. Grafik

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
m = [summary.loc[v, "accuracy_mean"] * 100 for v in VARIANTS]
s = [summary.loc[v, "accuracy_std"] * 100 for v in VARIANTS]
ax[0].bar(VARIANTS, m, yerr=s, capsize=4, color=["#8C8494", "#3E8E63", "#B5294E", "#3D6FB6", "#D98E04"])
for i, val in enumerate(m):
    ax[0].text(i, val + 0.3, f"{val:.2f}", ha="center")
ax[0].set_ylabel("Akurasi (%)"); ax[0].set_title("Ablasi: rata-rata akurasi ± SD (30 seed)")
ax[1].boxplot([piv[v] * 100 for v in VARIANTS])
ax[1].set_xticks(range(1, len(VARIANTS) + 1), VARIANTS)
ax[1].set_ylabel("Akurasi (%)"); ax[1].set_title("Sebaran akurasi per varian")
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150)
plt.show()

## 14. Matriks kebingungan (run dengan fungsi objektif terkecil)

In [ ]:
def cm_table(v):
    lab, mp = best[v]["labels"], best[v]["mapping"]
    pred = np.array([CLASS_NAMES[mp[l]] for l in lab])
    return pd.crosstab(pd.Series([CLASS_NAMES[i] for i in y_test], name="Aktual"),
                       pd.Series(pred, name="Prediksi"))
display(cm_table("E"))

## 15. Simpan hasil

In [ ]:
with pd.ExcelWriter(OUTPUT_EXCEL) as w:
    df.to_excel(w, sheet_name="per_run", index=False)
    summary.to_excel(w, sheet_name="ringkasan")
    tab.to_excel(w, sheet_name="tabel_ablasi", index=False)
    ttest.to_excel(w, sheet_name="uji_t", index=False)
    cm_table("A").to_excel(w, sheet_name="confusion_A")
    cm_table("E").to_excel(w, sheet_name="confusion_E")
    pd.DataFrame({"parameter": ["K", "N_RUNS", "MASTER_SEED", "N_BINS", "FDK_INTENS", "CANNY",
                                "LAMBDA_NUM", "CJ_MODE", "CJ_M", "MAX_ITER"],
                  "nilai": [K, N_RUNS, MASTER_SEED, N_BINS, str(FDK_INTENS),
                            f"{CANNY_LOW}/{CANNY_HIGH}", LAMBDA_NUM, CJ_MODE, CJ_M, MAX_ITER]}
                 ).to_excel(w, sheet_name="konfigurasi", index=False)
print("Tersimpan:", OUTPUT_EXCEL)